# 21 — Extraction frames 360° + sélection diversifiée + pré-annotation CVAT (cubemap)**Entrée :** 8 vidéos GoPro Max 360° équirectangulaires (`GS017063` → `GS017070`, 3072×1536, ratio 2.0)**Objectif :** produire **150 frames** diversifiées + GT pré-annotée pour fine-tuner Mask2Former.**Décisions issues de l'analyse pré-extraction :**- `GS017063` exclue (59 % frames floues, durée 18 s, faible diversité)- **Blur gate** (variance Laplacien ≥ seuil) — critique pour `GS017067` (28 % flou) et `GS017068` (21 % flou)- **Quota par vidéo** pondéré diversité × durée (pas une sélection globale unique)- Résolution annotation : **3072×1536** (précision pixel max)**Pipeline :**1. Extraction candidate par vidéo (sous-échantillonnage + blur gate)2. Sélection diversifiée gloutonne **par quota vidéo** (histogrammes couleur)3. Dédup global inter-vidéos (suppression quasi-doublons)4. Pré-annotation **via cubemap** (e2c → seg 6 faces → c2e) avec le **Mask2Former fine-tuné**5. Génération `cvat_preannotations.xml` (polygones, 19 classes Cityscapes)> **Pourquoi cubemap pour la pré-annotation ?** Le modèle fine-tuné a été entraîné sur des frames **perspective**, pas sur la distorsion équirectangulaire. L'inférence directe sur la panoramique donne des masques bruités. La pipeline cubemap segmente 6 faces perspective (que le modèle comprend) puis recompose — pré-annotations propres = correction CVAT rapide.

## 1. Environnement + chemins

In [ ]:
import sys, torch, time, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
import mmseg
print(f"mmseg   : {mmseg.__version__}")

try:
    import py360convert
    print("py360convert : OK")
except ImportError:
    raise ImportError("py360convert manquant : pip install py360convert")

PROJECT_ROOT    = Path("..").resolve()
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints"

# Vidéos 360° équirectangulaires
VIDEO_DIR = Path(r"C:\\Users\\youss\\Videos")
PATTERN   = "GS01706*_equirect.mp4"

# Sorties
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs" / "video360_annotation"
FRAMES_DIR = OUTPUT_DIR / "candidate_frames"      # toutes les frames candidates
ANNOT_DIR  = OUTPUT_DIR / "frames_to_annotate"    # 150 frames sélectionnées (upload CVAT)
for d in (OUTPUT_DIR, FRAMES_DIR, ANNOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"\nVidéos  : {VIDEO_DIR}")
print(f"Sortie  : {OUTPUT_DIR}")

## 2. Paramètres extraction & quotasQuotas par vidéo issus de l'analyse pré-extraction (pondération diversité × durée, total = 150).`GS017063` exclue.

In [ ]:
# Budget total d'annotation
N_ANNOT_TOTAL = 150

# Quota par vidéo (analyse pré-extraction — diversité × durée)
# GS017063 exclue (floue)
PER_VIDEO_QUOTA = {
    "GS017064[1]_equirect.mp4": 12,
    "GS017065[1]_equirect.mp4": 17,
    "GS017066[1]_equirect.mp4": 35,
    "GS017067[1]_equirect.mp4": 11,
    "GS017068[1]_equirect.mp4": 13,
    "GS017069[1]_equirect.mp4": 24,
    "GS017070[1]_equirect.mp4": 38,
}
assert sum(PER_VIDEO_QUOTA.values()) == N_ANNOT_TOTAL

# Extraction candidate
CANDIDATE_FPS = 0.5      # 1 frame / 2 s comme pool candidat (large, on filtre ensuite)
BLUR_THRESHOLD = 100.0   # variance Laplacien : frames en dessous = rejetées (floues)
# on garde ~4× le quota en candidats pour laisser le choix à la sélection diversifiée
CANDIDATE_MULT = 4

print("Quotas par vidéo :")
for k, v in PER_VIDEO_QUOTA.items():
    print(f"  {k:<30s} {v:>3d}")
print(f"  {'TOTAL':<30s} {sum(PER_VIDEO_QUOTA.values()):>3d}")

## 3. Extraction des frames candidates (avec blur gate)Pour chaque vidéo : sous-échantillonnage à `CANDIDATE_FPS`, rejet des frames sous le seuil de netteté.On conserve un pool de candidats (~4× le quota) parmi lesquels l'étape 4 choisira les plus diversifiées.

In [ ]:
def laplacian_var(bgr):
    """Netteté : variance du Laplacien (sur niveaux de gris réduits pour la vitesse)."""
    g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (768, 384))   # réduit = plus rapide, ordre de grandeur conservé
    return cv2.Laplacian(g, cv2.CV_64F).var()


def extract_candidates(video_path, quota):
    """Extrait des frames candidates nettes, sous-échantillonnées."""
    cap = cv2.VideoCapture(str(video_path))
    src_fps = cap.get(cv2.CAP_PROP_FPS)
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step    = max(1, round(src_fps / CANDIDATE_FPS))
    target_candidates = quota * CANDIDATE_MULT

    stem = video_path.stem
    saved, kept, rejected = 0, [], 0
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            sv = laplacian_var(frame)
            if sv >= BLUR_THRESHOLD:
                out = FRAMES_DIR / f"{stem}__f{idx:06d}.jpg"
                cv2.imwrite(str(out), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
                kept.append(out)
                saved += 1
            else:
                rejected += 1
        idx += 1
    cap.release()
    return kept, rejected


candidate_pool = {}   # stem -> [frame paths]
print(f"{'vidéo':<30s}{'candidats':>10s}{'rejetés(flou)':>15s}")
print("-" * 55)
for fname, quota in PER_VIDEO_QUOTA.items():
    vp = VIDEO_DIR / fname
    assert vp.exists(), f"Vidéo introuvable : {vp}"
    kept, rejected = extract_candidates(vp, quota)
    candidate_pool[vp.stem] = kept
    print(f"{fname:<30s}{len(kept):>10d}{rejected:>15d}")

total_candidates = sum(len(v) for v in candidate_pool.values())
print(f"\n{total_candidates} frames candidates extraites dans {FRAMES_DIR}")

## 4. Sélection diversifiée par quota vidéoSélection gloutonne (distance Bhattacharyya entre histogrammes couleur) **à l'intérieur du quota de chaque vidéo** : on prend itérativement la frame la plus différente de celles déjà retenues. Même métrique que le notebook 17, appliquée par vidéo.

In [ ]:
def color_hist(path):
    img = cv2.imread(str(path))
    img = cv2.resize(img, (256, 256))
    h = cv2.calcHist([img], [0, 1, 2], None, [8, 8, 8], [0, 256] * 3)
    return cv2.normalize(h, h).flatten()


def greedy_diverse(frames, k):
    """Sélectionne k frames les plus diversifiées (glouton sur histogrammes)."""
    if len(frames) <= k:
        return list(frames)
    hists = {f: color_hist(f) for f in frames}
    # graine = frame la plus "centrale" inversée -> on part de la première, robuste
    selected = [frames[0]]
    while len(selected) < k:
        best_f, best_d = None, -1
        for f in frames:
            if f in selected:
                continue
            d = min(cv2.compareHist(hists[f], hists[s], cv2.HISTCMP_BHATTACHARYYA)
                    for s in selected)
            if d > best_d:
                best_d, best_f = d, f
        selected.append(best_f)
    return selected


selected_per_video = {}
print(f"{'vidéo':<30s}{'quota':>7s}{'dispo':>7s}{'choisi':>8s}")
print("-" * 52)
for fname, quota in PER_VIDEO_QUOTA.items():
    stem = (VIDEO_DIR / fname).stem
    pool = candidate_pool[stem]
    sel = greedy_diverse(pool, quota)
    selected_per_video[stem] = sel
    print(f"{fname:<30s}{quota:>7d}{len(pool):>7d}{len(sel):>8d}")

all_selected = [f for sel in selected_per_video.values() for f in sel]
print(f"\n{len(all_selected)} frames sélectionnées (avant dédup inter-vidéos)")

## 5. Déduplication inter-vidéosSuppression des quasi-doublons entre vidéos (deux scènes très proches issues de vidéos différentes). Si une frame est trop similaire à une déjà gardée (distance < seuil), elle est remplacée par la candidate diversifiée suivante de sa vidéo.

In [ ]:
DEDUP_THRESHOLD = 0.15   # distance Bhattacharyya min entre 2 frames retenues

hist_cache = {f: color_hist(f) for f in all_selected}

final, dropped = [], 0
for f in all_selected:
    too_close = any(
        cv2.compareHist(hist_cache[f], hist_cache[g], cv2.HISTCMP_BHATTACHARYYA) < DEDUP_THRESHOLD
        for g in final
    )
    if too_close:
        dropped += 1
    else:
        final.append(f)

print(f"Frames retenues après dédup : {len(final)} (supprimées : {dropped})")

# Copie vers le dossier d'annotation
for f in ANNOT_DIR.glob("*.jpg"):
    f.unlink()
for f in sorted(final):
    shutil.copy(f, ANNOT_DIR / f.name)

selected = sorted(ANNOT_DIR.glob("*.jpg"))
print(f"{len(selected)} frames copiées dans {ANNOT_DIR}")
print("Ce dossier est à uploader dans CVAT.")

## 6. Palette + labels Cityscapes (19 classes)

In [ ]:
CITYSCAPES_PALETTE = np.array([
    [128,  64, 128], [244,  35, 232], [ 70,  70,  70], [102, 102, 156], [190, 153, 153],
    [153, 153, 153], [250, 170,  30], [220, 220,   0], [107, 142,  35], [152, 251, 152],
    [ 70, 130, 180], [220,  20,  60], [255,   0,   0], [  0,   0, 142], [  0,   0,  70],
    [  0,  60, 100], [  0,  80, 100], [  0,   0, 230], [119,  11,  32],
], dtype=np.uint8)

CITYSCAPES_LABELS = [
    "road","sidewalk","building","wall","fence","pole","traffic_light",
    "traffic_sign","vegetation","terrain","sky","person","rider",
    "car","truck","bus","train","motorcycle","bicycle",
]
CITYSCAPES_HEX = ['#%02x%02x%02x' % tuple(c) for c in CITYSCAPES_PALETTE]
print(f"{len(CITYSCAPES_LABELS)} classes Cityscapes prêtes.")

## 7. Chargement du Mask2Former fine-tunéOn utilise le checkpoint fine-tuné CoolPath (sortie du notebook 18), pas le modèle Cityscapes de base.

In [ ]:
from mmseg.apis import init_model, inference_model

# Checkpoint fine-tuné (sortie notebook 18) — adapte si le nom diffère
FINETUNE_WORK_DIR = PROJECT_ROOT / "data" / "outputs" / "finetune_mask2former"
ft_cfg = FINETUNE_WORK_DIR / "mask2former_coolpath_finetune.py"

# cherche le dernier checkpoint .pth du fine-tuning
ft_ckpts = sorted(FINETUNE_WORK_DIR.glob("**/*.pth"))
assert ft_cfg.exists(), f"Config fine-tune introuvable : {ft_cfg}"
assert ft_ckpts, f"Aucun checkpoint fine-tuné dans {FINETUNE_WORK_DIR}"

# privilégie iter_*.pth le plus avancé / best_*.pth s'il existe
best = [c for c in ft_ckpts if "best" in c.name.lower()]
ft_ckpt = best[0] if best else ft_ckpts[-1]

print(f"Config     : {ft_cfg.name}")
print(f"Checkpoint : {ft_ckpt.name}")

model = init_model(str(ft_cfg), str(ft_ckpt), device="cuda:0")
print("Modèle fine-tuné chargé.")

## 8. Pré-annotation via pipeline cubemapPour chaque frame équirectangulaire :1. `e2c` → 6 faces cube (perspective, format que le modèle comprend)2. inférence Mask2Former sur chaque face3. `c2e` (`mode='nearest'`) → masque équirectangulaire recomposéLe masque final est aligné pixel-à-pixel avec la frame équirectangulaire d'origine.

In [ ]:
import py360convert

CUBE_FACE = 768   # résolution d'une face cube pour l'inférence (16 Go VRAM → confortable)

def segment_equirect_cubemap(equi_bgr):
    """Segmente une frame équirectangulaire via cubemap. Retourne mask équirect (H,W) uint8."""
    H, W = equi_bgr.shape[:2]
    equi_rgb = cv2.cvtColor(equi_bgr, cv2.COLOR_BGR2RGB)

    # e2c : dict des 6 faces (F,R,B,L,U,D)
    faces = py360convert.e2c(equi_rgb, face_w=CUBE_FACE, cube_format="dict")

    pred_faces = {}
    for k, face_rgb in faces.items():
        face_bgr = cv2.cvtColor(face_rgb.astype(np.uint8), cv2.COLOR_RGB2BGR)
        res = inference_model(model, face_bgr)
        pred = res.pred_sem_seg.data[0].cpu().numpy().astype(np.uint8)
        # py360convert c2e attend du float pour interp ; on garde nearest plus bas
        pred_faces[k] = pred

    # empile en (H,W,1) par face pour c2e, mode nearest préserve les labels
    # c2e attend cube_format dict de faces 2D -> on ajoute un canal puis on retire
    pred_faces_3d = {k: v[..., None].astype(np.float32) for k, v in pred_faces.items()}
    equi_mask = py360convert.c2e(pred_faces_3d, h=H, w=W,
                                 mode="nearest", cube_format="dict")
    equi_mask = np.round(equi_mask[..., 0]).astype(np.uint8)
    return equi_mask


# test rapide sur 1 frame
_t0 = time.perf_counter()
_test = cv2.imread(str(selected[0]))
_m = segment_equirect_cubemap(_test)
print(f"Test 1 frame : {_test.shape[:2]} → mask {_m.shape}, "
      f"{time.perf_counter()-_t0:.1f}s, classes={np.unique(_m)[:10]}")

## 9. Génération du XML CVAT (polygones, format CVAT for images 1.1)

In [ ]:
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom

root = ET.Element("annotations")
ET.SubElement(root, "version").text = "1.1"
meta = ET.SubElement(root, "meta")
task = ET.SubElement(meta, "task")
labels_el = ET.SubElement(task, "labels")
for lbl_name, hex_col in zip(CITYSCAPES_LABELS, CITYSCAPES_HEX):
    le = ET.SubElement(labels_el, "label")
    ET.SubElement(le, "name").text = lbl_name
    ET.SubElement(le, "color").text = hex_col
    ET.SubElement(le, "type").text = "polygon"
    ET.SubElement(le, "attributes")

MIN_CONTOUR_AREA = 600   # 360° = grandes frames, seuil un peu plus haut qu'en perspective

for img_id, fp in enumerate(selected):
    orig = cv2.imread(str(fp))
    H0, W0 = orig.shape[:2]

    mask = segment_equirect_cubemap(orig)   # même résolution que l'image

    img_el = ET.SubElement(root, "image", id=str(img_id), name=fp.name,
                           width=str(W0), height=str(H0))

    for cid in np.unique(mask):
        if cid >= 19:
            continue
        binary = (mask == cid).astype(np.uint8)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < MIN_CONTOUR_AREA:
                continue
            eps = 0.002 * cv2.arcLength(cnt, True)
            approx = cv2.approxPolyDP(cnt, eps, True)
            if len(approx) < 3:
                continue
            pts = ";".join(f"{p[0][0]:.1f},{p[0][1]:.1f}" for p in approx)
            ET.SubElement(img_el, "polygon", label=CITYSCAPES_LABELS[cid],
                          source="auto", occluded="0", points=pts, z_order="0")

    if (img_id + 1) % 10 == 0 or img_id == len(selected) - 1:
        print(f"  [{img_id+1}/{len(selected)}] pré-annotées")

del model
torch.cuda.empty_cache()

xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")
xml_path = OUTPUT_DIR / "cvat_preannotations.xml"
xml_path.write_text(xml_str, encoding="utf-8")
print(f"\nPré-annotations CVAT : {xml_path}")
print(f"{len(selected)} images, polygones générés (cubemap + Mask2Former fine-tuné)")

## 10. Label set CVAT (JSON importable)

In [ ]:
import json
cvat_labels = [
    {"name": n, "color": c, "type": "polygon", "attributes": []}
    for n, c in zip(CITYSCAPES_LABELS, CITYSCAPES_HEX)
]
labels_path = OUTPUT_DIR / "cvat_labels_cityscapes19.json"
labels_path.write_text(json.dumps(cvat_labels, indent=2), encoding="utf-8")
print(f"Label set CVAT : {labels_path}")

## 11. Récapitulatif & workflow CVAT**Fichiers produits :**- `frames_to_annotate/` — 150 frames équirectangulaires à uploader- `cvat_preannotations.xml` — pré-annotations polygones (à importer après upload)- `cvat_labels_cityscapes19.json` — label set (à coller dans l'éditeur de labels, bouton *Raw*)**Étapes CVAT :**1. Créer une tâche `coolpath_360_annotation`, importer le JSON de labels, uploader `frames_to_annotate/`2. *Actions → Upload annotations* → format **CVAT for images 1.1** → `cvat_preannotations.xml`3. Corriger les polygones (la distorsion équirectangulaire demande surtout d'ajuster horizon courbe + zone porteur au nadir). Activer **SAM** pour les contours fins.4. *Actions → Export annotations* → **Segmentation mask 1.1**5. Brancher l'export dans le notebook 18 (fine-tuning) — passer `CVAT_DIR` sur ce nouvel export et relancer.**Fine-tuning haute précision (rappel) :**- Entraîner en crops **1024×1024** (16 Go VRAM le permet) → meilleurs bords pixel- TTA multi-échelle (0.75–2.0 + flip) à l'inférence — gain gratuit- Depth Anything V2 pour corriger le biais saisonnier (feuillage hallucination → ciel)